# CS336 — lecture_01

Converted from the original Stanford executable lecture.

In [1]:
import os
import regex
import tiktoken
from abc import ABC
from dataclasses import dataclass
from collections import defaultdict

## Definitions

Classes and helper functions used by the following examples.

In [2]:
class Tokenizer(ABC):
    """Abstract interface for a tokenizer."""
    def encode(self, string: str) -> list[int]:
        raise NotImplementedError

    def decode(self, indices: list[int]) -> str:
        raise NotImplementedError


class CharacterTokenizer(Tokenizer):
    """Represent a string as a sequence of Unicode code points."""
    def encode(self, string: str) -> list[int]:
        return list(map(ord, string))

    def decode(self, indices: list[int]) -> str:
        return "".join(map(chr, indices))


class ByteTokenizer(Tokenizer):
    """Represent a string as a sequence of bytes."""
    def encode(self, string: str) -> list[int]:
        string_bytes = string.encode("utf-8")  # @inspect string_bytes
        indices = list(map(int, string_bytes))  # @inspect indices
        return indices

    def decode(self, indices: list[int]) -> str:
        string_bytes = bytes(indices)  # @inspect string_bytes
        string = string_bytes.decode("utf-8")  # @inspect string
        return string


def merge(indices: list[int], pair: tuple[int, int], new_index: int) -> list[int]:  # @inspect indices, @inspect pair, @inspect new_index
    """Return `indices`, but with all instances of `pair` replaced with `new_index`."""
    new_indices = []  # @inspect new_indices
    i = 0  # @inspect i
    while i < len(indices):
        if i + 1 < len(indices) and indices[i] == pair[0] and indices[i + 1] == pair[1]:
            new_indices.append(new_index)
            i += 2
        else:
            new_indices.append(indices[i])
            i += 1
    return new_indices


@dataclass(frozen=True)
class BPETokenizerParams:
    """All you need to specify a BPETokenizer."""
    vocab: dict[int, bytes]     # index -> bytes
    merges: dict[tuple[int, int], int]  # index1,index2 -> new_index


class BPETokenizer(Tokenizer):
    """BPE tokenizer given a set of merges and a vocabulary."""
    def __init__(self, params: BPETokenizerParams):
        self.params = params

    def encode(self, string: str) -> list[int]:
        indices = list(map(int, string.encode("utf-8")))  # @inspect indices
        # Note: this is a very slow implementation
        for pair, new_index in self.params.merges.items():  # @inspect pair, @inspect new_index
            indices = merge(indices, pair, new_index)  # @stepover
        return indices

    def decode(self, indices: list[int]) -> str:
        bytes_list = list(map(self.params.vocab.get, indices))  # @inspect bytes_list
        string = b"".join(bytes_list).decode("utf-8")  # @inspect string
        return string


def get_compression_ratio(string: str, indices: list[int]) -> float:  # @inspect string indices
    """Given `string` that has been tokenized into `indices`, return the number of UTF-8 bytes per token.."""
    num_bytes = len(bytes(string, encoding="utf-8"))  # @inspect num_bytes
    num_tokens = len(indices)                       # @inspect num_tokens
    return num_bytes / num_tokens


def get_gpt5_tokenizer():
    # Code: https://github.com/openai/tiktoken
    return tiktoken.get_encoding("o200k_base")


def output_tokenizer(tokenizer, path: str):
    """Write out the vocabulary of `tokenizer` to `path`, one per line."""
    if not os.path.exists(path):
        vocab = [b.decode("utf-8", errors="replace") for b in tokenizer.token_byte_values()]
        with open(path, "w") as f:
            for token in vocab:
                f.write(token + "\n")


def train_bpe(string: str, num_merges: int) -> BPETokenizerParams:
    indices = list(map(int, string.encode('utf-8')))
    merges: dict[tuple[int, int], int] = {}
    vocab: dict[int, bytes] = {x: bytes([x]) for x in range(256)}
    for i in range(num_merges):
        counts = count_adjacent_pairs(indices)
        pair = max(counts, key=counts.get)
        new_index = 256 + i
        merges[pair] = new_index
        vocab[new_index] = vocab[pair[0]] + vocab[pair[1]]
        indices = merge(indices, pair, new_index)
    compression_ratio = get_compression_ratio(string, indices)
    return BPETokenizerParams(vocab=vocab, merges=merges)


def count_adjacent_pairs(indices: list[int]) -> dict[tuple[int, int], int]:
    """Return a dictionary mapping each adjacent pair of tokens in `indices` to the number of times it occurs."""
    counts = defaultdict(int)
    for index1, index2 in zip(indices, indices[1:]):
        counts[(index1, index2)] += 1
    return counts

## CS336: Language Models From Scratch (Spring 2026)

![Lecture image](images/course-staff.png)

...bringing you the 3rd offering of CS336.

Lectures from 2nd offering (Spring 2025) are on [YouTube](https://www.youtube.com/playlist?list=PLoROMvodv4rOY23Y0BoGoBGgQ1zmU_MT_).

What's new?

- Same 'from scratch' philosophy

- Prioritize high value-per-time concepts, don't lose the forest for the trees

- More coverage of modern LM ingredients (mixture of experts, long-context, agents)

## Why did we make this course?

Problem: researchers are becoming **disconnected** from the underlying technology.

- 2016: researchers implemented and trained their own models.

- 2018: researchers downloaded models (e.g., BERT) and fine-tuned them.

- Today: researchers prompt API models (e.g., GPT/Claude/Gemini).

Moving up levels of abstraction boosts productivity, but

- These abstractions are leaky (in contrast to programming languages or operating systems).

- There is still fundamental research to be done that requires tearing up the stack.

**Full understanding** of this technology is necessary for **fundamental research**.

Philosophy of this course: **understanding via building**.

But there's one small problem...

## The industrialization of language models

![Lecture image](https://upload.wikimedia.org/wikipedia/commons/c/cc/Industrialisation.jpg)

Frontier models are really expensive:

- 2023: GPT-4 supposedly cost $100M to train.  [https://www.wired.com/story/openai-ceo-sam-altman-the-age-of-giant-ai-models-is-already-over/](https://www.wired.com/story/openai-ceo-sam-altman-the-age-of-giant-ai-models-is-already-over/)

- 2025: xAI builds cluster with 230K GPUs for training Grok.  [https://x.com/elonmusk/status/1947701807389515912](https://x.com/elonmusk/status/1947701807389515912)

There are no public details on how frontier models are built.

From the GPT-4 technical report  *[gpt_4_2023]*:

![Lecture image](images/gpt4-no-details.png)

Frontier models are out of reach for us.

We could build small language models (<1B parameters), but this might not be representative of large language models.

Example 1: fraction of FLOPs spent in attention versus MLP changes with scale.  [https://x.com/stephenroller/status/1579993017234382849](https://x.com/stephenroller/status/1579993017234382849)

![Lecture image](images/roller-flops.png)

Example 2: emergence of behavior with scale  [https://arxiv.org/pdf/2206.07682](https://arxiv.org/pdf/2206.07682)

![Lecture image](images/wei-emergence-plot.png)

## What can we learn in this class that transfers to frontier models?

There are three types of knowledge:

- **Mechanics**: how things work (what a Transformer is, how model parallelism works)

- **Mindset**: squeezing the most out of the hardware, taking scaling seriously

- **Intuitions**: which data and modeling decisions yield good accuracy

We can teach mechanics and mindset (these do transfer).

We can only partially teach intuitions (do not necessarily transfer across scales).

## Intuitions? 🤷

Some design decisions are simply not (yet) justifiable and just come from experimentation.

Example: Noam Shazeer paper that introduced SwiGLU  *[shazeer_2020]*

![Lecture image](images/divine-benevolence.png)

## The bitter lesson

Wrong interpretation: scale is all that matters, algorithms don't matter.

Right interpretation: algorithms that scale are what matter.

### accuracy = efficiency x resources

In fact, efficiency is way more important at larger scales (can't afford to be wasteful).

 [https://arxiv.org/abs/2005.04305](https://arxiv.org/abs/2005.04305) showed 44x algorithmic efficiency on ImageNet between 2012 and 2019.

Framing: what is the best model one can build given a certain compute and data budget?

In other words, **maximize efficiency**!

## Pre-neural (before 2010s)

- Language model to measure the entropy of English  *[shannon_1950]*

- N-gram language models (used in machine translation and speech recognition systems)  *[brants_2007]*

## Neural ingredients (2010s)

- Long-Short Term Memory (LSTM)  *[lstm_1997]*

- First neural language model  *[bengio_2003]*

- Sequence-to-sequence modeling (for machine translation)  *[seq2seq_2014]*

- Adam optimizer  *[adam_2014]*

- Attention mechanism (for machine translation)  *[bahdanau_2015_attention]*

- Transformer architecture (for machine translation)  *[transformer_2017]*

- Mixture of experts  *[moe_2017]*

- Model parallelism  *[gpipe_2018]* *[zero_2019]* *[megatron_lm_2019]*

## Early foundation models (late 2010s)

- ELMo: pretraining with LSTMs, fine-tuning improves downstream tasks  *[elmo_2018]*

- BERT: pretraining with Transformer, fine-tuning improves downstream tasks  *[bert_2018]*

- Google's T5 (11B): cast everything as text-to-text  *[t5_2019]*

## Embracing scaling

- OpenAI's GPT-2 (1.5B): fluent text, first signs of zero-shot  *[gpt2_2019]*

- Scaling laws: provide hope / predictability for scaling  *[kaplan_scaling_laws_2020]*

- OpenAI's GPT-3 (175B): in-context learning  *[gpt_3_2020]*

- Google's PaLM (540B): massive scale, undertrained  *[palm_2022]*

- DeepMind's Chinchilla (70B): compute-optimal scaling laws  *[chinchilla_2022]*

## Open models

Early attempts (attempts to replicate GPT-3):

- EleutherAI's open datasets (The Pile) and models (GPT-J)  *[the_pile_2020]* *[gpt_j_2021]*

- Meta's OPT (175B): GPT-3 replication, lots of hardware issues  *[opt_175b_2022]*

- Hugging Face / BigScience's BLOOM (176B): focused on data sourcing  *[bloom_2022]*

Credible open-weight models (weights + paper):

- Meta's Llama models  *[llama_2023]* *[llama_2_2023]* *[llama_3_2024]*

- Mistral's models  *[mistral_7b_2023]* *[mixtral_2024]*

- DeepSeek's models  *[deepseek_67b_2024]* *[deepseek_v2_2024]* *[deepseek_v3_2024]* *[deepseek_r1_2025]* *[deepseek_v3_2_2025]*

- Alibaba's Qwen models  *[qwen_2_5_2024]* *[qwen_3_2025]*

- Moonshot's Kimi models  *[kimi_1_5_2025]* *[kimi_k2_5_2026]*

- Z.ai's GLM models  *[glm_4_5_2025]* *[glm_5_2026]*

- Minimax's models  *[minimax_m2_5_2026]*

- Xiaomi's MIMO models  *[xiaomi_mimo_v2_2026]*

These models are approaching closed models (GPT, Claude, Gemini, etc.).

Open-source models (weights + paper + code + data):

- AI2's Olmo models  *[olmo_7b_2024]* *[olmo_2_2025]* *[olmo_3_2025]*

- NVIDIA's Nemotron models  *[nemotron_15b_2024]* *[nemotron_3_2025]*

- Marin's models (open development)  *[marin_8b_2025]* *[marin_32b_2025]*

Openness is important for trust and innovation  [https://arxiv.org/abs/2403.07918](https://arxiv.org/abs/2403.07918)

Ideas from open models enable us to teach CS336.

What is a language model?

- 2018 (BERT): something you fine-tune

- 2020 (GPT-3): something you prompt

- 2022 (ChatGPT): something you talk to  [example conversation](https://huggingface.co/datasets/HuggingFaceTB/smoltalk/viewer/all/train?row=72&conversation-viewer=72)

- 2026 (agents): something that acts autonomously  [example trace](https://huggingface.co/datasets/nebius/SWE-rebench-openhands-trajectories/viewer/default/train?conversation-viewer=1)

The fundamentals are the same (attention, kernels, optimization).

The specs are different (longer context, inference efficiency matters even more).

This is an *executable lecture*, a program whose execution delivers the content of a lecture.

Executable lectures make it possible to:

- view and run code (since everything is code!),

In [3]:
total = 0  # @inspect total

for x in [1, 2, 3]:  # @inspect x
    total += x  # @inspect total

- see the hierarchical structure of the lecture

All information online:  [course website](https://stanford-cs336.github.io/spring2026/)

This is a 5-unit class.

Comment from Spring 2024 course evaluation:

> *The entire assignment was approximately the same amount of work as all 5 assignments from CS 224n plus the final project. And that's just the first homework assignment.*

## Why you should take this course

- You have an obsessive need to understand how things work.

- You want to build up your research engineering muscles.

## Why you should not take this course

- You actually want to get research done this quarter. (Talk to your advisor.)

- You are interested in learning about the hottest new techniques in AI (e.g., multimodality, RAG, etc.). (You should take a seminar class for that.)

- You want to get good results on your own application domain. (You should just prompt or fine-tune an existing model.)

## How you can follow along at home

- All lecture materials and assignments will be posted online, so feel free to follow on your own.

- Lectures are recorded via [CGOE](https://cgoe.stanford.edu/).

## Assignments

- 5 assignments (basics, systems, scaling laws, data, alignment).

- No scaffolding code, but we provide unit tests and adapter interfaces to help you check correctness.

- Implement locally to test for correctness, then run on cluster for benchmarking (accuracy and speed).

- Leaderboard for some assignments (minimize perplexity given training budget).

## AI policy

- Coding agents can solve all the assignments, but you won't learn anything.

- AI can be tremendously useful for answering questions and tutoring.

- You must use our provided AGENTS.md file, which asks the AI to be pedagogically-minded.

- Please read our [AI policy guide](https://docs.google.com/document/d/1SZAlExB1qAc9izHt54gwunNpjKE6wXb8Y7yA_e-baK8/edit?tab=t.0).

## Compute

- Thanks to [Modal](https://modal.com/) for providing compute. 🙏

- Please read the [guide](https://docs.google.com/document/d/1cHE0iKVyXLJ3XpIs2XuXTmZ-HMmPk2hIPeCvy-AydMg/edit?tab=t.otis27tacaef) on how to access and use the compute.

Goal: be able to train a basic language model

Components: tokenization, model architecture, training

## Tokenization

What are the atoms that the model operates on?

Formally: a tokenizer converts between raw inputs (bytes) and sequences of integers (tokens)

![Lecture image](images/tokenized-example.png)

Popular tokenizer: **Byte-Pair Encoding** (BPE)  *[sennrich_2016]*

Intuition: break input into frequently-occuring chunks

Efficiency lens

- Reduce context length (1000 bytes → ~250 tokens)

- Adaptive computation (more modeling capacity on interesting parts of input)

The dream: tokenizer-free model architectures, which operate directly on bytes  *[byt5_2021]* *[megabyte_2023]* *[blt_2024]* *[tfree_2024]* *[hnet_2025]*

These are promising, but have not yet been scaled up to the frontier.

## Model architecture

Starting point: original Transformer  *[transformer_2017]*

![Lecture image](images/transformer-architecture.png)

Refinements:

- Activation functions: ReLU, SwiGLU  *[shazeer_2020]*

- Positional encodings: sinusoidal, RoPE  *[rope_2021]*

- Normalization: LayerNorm, RMSNorm, QK norm, pre-norm versus post-norm  *[layernorm_2016]* *[rms_norm_2019]* *[qk_norm_2023]* *[pre_post_norm_2020]*

- Attention: full, sparse/local attention, group-query attention (GQA), multi-head latent attention (MLA)  *[sparse_transformer_2019]* *[gqa_2023]* *[mla_2024]*

- Recurrence/state-space models/linear attention: Mamba, Gated DeltaNet  *[linear_attention_2020]* *[mamba_2_2024]* *[gdn_2024]* *[mamba_3_2026]*

- MLP: dense, mixture of experts  *[moe_2017]* *[switch_transformers_2021]*

- Shape (hidden dimension, depth, number of heads, number of experts)

## Training

How do you set the parameters of the model?

- Loss function (e.g., multi-token prediction)  *[mtp_2024]* *[deepseek_v3_2024]*

- Optimizer (e.g., AdamW, SOAP, Muon)  *[adam_2014]* *[adamw_2017]* *[soap_2024]* *[muon_2024]*

- Initialization scale (e.g., Xavier init, muP)  *[glorot_2010]* *[mup_2022]*

- Learning rate schedule (e.g., cosine, WSD)  *[cosine_learning_rate_2017]* *[wsd_2024]*

- Regularization (e.g., dropout, weight decay)

- Batch size (e.g., critical batch size)  *[large_batch_training_2018]*

- MoE specific: load balancing (e.g., aux-free)  *[auxfree_2024]* *[deepseek_v3_2024]*

## Assignment 1 (basics)

 [GitHub](https://github.com/stanford-cs336/assignment1-basics) [PDF](https://github.com/stanford-cs336/assignment1-basics/blob/main/cs336_spring2026_assignment1_basics.pdf)

- Implement BPE tokenizer

- Implement Transformer, cross-entropy loss, AdamW optimizer, training loop

- Do resource accounting

- Train on TinyStories and OpenWebText

- Leaderboard: minimize OpenWebText perplexity given 45 minutes on a B200  [last year's leaderboard](https://github.com/stanford-cs336/spring2025-assignment1-basics-leaderboard)

High-level principle: everything is about balancing the following:

- Expressivity (can represent complex dependencies in the data)

- Stability (keep parameter and gradient norms in goldilocks zone)

- Efficiency (runs fast on hardware, both training and inference)

Goal: squeeze the most out of the hardware (GPU or TPU)

Components: kernels, parallelism, inference

## Basics

- Resource accounting: memory and compute characteristics of a model

In [4]:
total_flops = 6 * 70e9 * 1e12  # Training 70B parameters on 1T tokens = 4.2e23 FLOPs @inspect total_flops

![Lecture image](images/compute-memory.png)

- Model parameters must be moved from memory (HBM) to the compute (SMs)

- Example: B200 can perform 2.25 PFLOP/sec (bf16) with 8TB/sec memory bandwidth

- Roofline analysis: understand whether we're compute-bound or memory-bound

- Benchmarking and profiling (nsight): see what happens in practice

[DGX B200](https://docs.nvidia.com/dgx/dgxb200-user-guide/introduction-to-dgxb200.html):

![Lecture image](https://docs.nvidia.com/dgx/dgxb200-user-guide/_images/dgx-b200-system-topology.png)

## Kernels

- Kernel is a function that runs on GPU

- When using PyTorch, each primitive operation launches a standard kernel

- Can write custom kernels to make GPUs go brrr

- Principle: organize computation to minimize data movement

- Naive: read HBM; compute A; write HBM; read HBM; compute B; write HBM

- Fused: read HBM; compute A and B; write HBM

- Strategies: operator fusion (matmul + activation), tiling (FlashAttention)

- Warp divergence, memory coalescing, bank conflicts, occupancy, bulk-async memory transfers

- Write kernels in CUDA/**Triton**/CUTLASS/ThunderKittens

## Parallelism

- What if we have 1024 GPUs?

- Data movement between GPUs is even slower, but same 'minimize data movement' principle holds

- Use classic collective operations (e.g., gather, reduce, all-reduce)

- Shard memory (parameters, activations, gradients, optimizer states) across GPUs

- How to split computation: {data,tensor,pipeline,sequence,expert} parallelism

## Inference

Goal: generate tokens given a prompt (needed to actually use models!)

Inference is also needed for reinforcement learning, test-time compute, evaluation

Two phases: prefill and decode

![Lecture image](images/prefill-decode.png)

- Prefill (similar to training): tokens are given, can process all at once (compute-bound)

- Decode: need to generate one token at a time (memory-bound)

Methods to speed up decoding:

- Use cheaper model (via model pruning, quantization, distillation)

- Speculative decoding: use a cheaper "draft" model to generate multiple tokens, then use the full model to score in parallel (exact decoding!)

- Systems optimizations: fused kernels, continuous batching

## Assignment 2 (systems)

 [GitHub](https://github.com/stanford-cs336/assignment2-systems) [PDF from Spring 2025](https://github.com/stanford-cs336/assignment2-systems/blob/spring2025/cs336_spring2025_assignment2_systems.pdf)

- Implement a fused RMSNorm kernel in Triton

- Implement distributed data parallel training

- Implement optimizer state sharding

- Benchmark and profile the implementations

Recommended book: [How to Scale Your Model](https://jax-ml.github.io/scaling-book/)

- Nicely lays out how to approach systems for LLMs conceptually

- From Google, so it foregrounds TPUs, but high-level concepts are similar

Setting: if you had 1e25 FLOPs of compute, what hyperparameters would you use to train a good model?

Too expensive to do hyperparameter tuning at full scale!

Key conceptual shift: instead of a single scale, think of a **scaling recipe** (FLOPs → hyperparameters)

For a scaling recipe:

- Run experiments to compute the loss at various smaller scales (e.g., up to 1e24 FLOPs)

- Fit a scaling law to predict the loss of the scaling recipe at the target scale (e.g., 1e25 FLOPs)

Now you can:

1. Optimize the scaling recipe targeting a larger scale using smaller scale experiments

2. Predict the loss at the target scale before actually running the experiment!

Scaling laws don't happen automatically, they require careful construction of a scaling recipe.

Parameterize the model in a way to get **hyperparameter transfer**  *[mup_2022]*

Predictability is at least as important as optimality!

Question: given a FLOPs budget (C = 6 N D), use a bigger model (N) or train on more tokens (D)?

Classic compute-optimal scaling laws:  *[kaplan_scaling_laws_2020]* *[chinchilla_2022]*

- ISOFLOP curves: for multiple small FLOPs budgets, find optimal N

- Then fit a scaling law to extrapolate to large FLOPs budgets

![Lecture image](images/chinchilla-isoflop.png)

TL;DR: D = 20 N is roughly optimal (e.g., 70B parameter model should be trained on ~1.4T tokens)

Caveat: this doesn't take into account inference costs (want a smaller model)

Live example from Marin  [https://x.com/percyliang/status/2034367256277533100](https://x.com/percyliang/status/2034367256277533100)

![Lecture image](https://pbs.twimg.com/media/HDuErvvbsAAQ5Yt?format=jpg&name=4096x4096)

Should be done training this week, should see how well we match the preregistered loss!

## Assignment 3 (scaling laws)

 [GitHub](https://github.com/stanford-cs336/assignment3-scaling) [PDF from Spring 2025](https://github.com/stanford-cs336/assignment3-scaling/blob/master/cs336_spring2025_assignment3_scaling.pdf)

- We define a training API (hyperparameters → loss) based on previous runs

- Submit "training jobs" (under a FLOPs budget) and gather data points

- Fit scaling laws to the data points

- Submit extrapolated hyperparameters and loss predictions

- Leaderboard: minimize loss given FLOPs budget

Question: What capabilities do we want the model to have?

Multilingual?  Good at conversation?  Agentic coding capabilities?

## Evaluation

What is the purpose of evaluation?

1. Internal: guide model development (smoothness across scales, relative performance matters)

2. External: measure absolute quality of a real use case (ecological validity matters)

Examples of evaluations:

1. Perplexity: ideally run on private documents not on Internet (avoid contamination)

2. Advanced use cases: GPQA, HLE, SWE-Bench, Terminal-Bench

LMs are general purpose, require a diverse set of evaluations!

## Data curation

- Data does not just fall from the sky.

- Sources: webpages crawled from the Internet, books, arXiv papers, GitHub code, etc.

![Lecture image](https://ar5iv.labs.arxiv.org/html/2101.00027/assets/pile_chart2.png)

- Appeal to fair use to train on copyright data?  [https://arxiv.org/pdf/2303.15715.pdf](https://arxiv.org/pdf/2303.15715.pdf)

- Might have to license data (e.g., Google with Reddit data)  [https://www.reuters.com/technology/reddit-ai-content-licensing-deal-with-google-sources-say-2024-02-22/](https://www.reuters.com/technology/reddit-ai-content-licensing-deal-with-google-sources-say-2024-02-22/)

- Raw data is HTML, PDF, directories (not text), requires processing

## Data processing

- Transformation: convert HTML/PDF to text (extract main content)

- Filtering: keep high quality data, remove harmful content (via classifiers)

- Deduplication: save compute, avoid memorization; use Bloom filters or MinHash

- Data mixing: how much to upweight/downweight each source?  *[regmix_2025]* *[olmix_2026]*

- Rewriting / synthetic data: use LM to augment real data, more similar to downstream tasks  *[wrap_2024]*

Types of data:

- Pretraining data: large and diverse

- Mid-training data: high quality, including long-context

- Post-training data: supervised fine-tuning (conversations, agentic traces with tool calling)

## Assignment 4 (data)

 [GitHub](https://github.com/stanford-cs336/assignment4-data) [PDF from Spring 2025](https://github.com/stanford-cs336/assignment4-data/blob/spring2025/cs336_spring2025_assignment4_data.pdf)

- Convert Common Crawl HTML to text

- Train classifiers to filter for quality and harmful content

- Deduplication using MinHash

- Leaderboard: minimize perplexity given token budget

So far, we have trained a model on full supervision (predict the next token).

Now that the model should be reasonable, we can improve it further from **weak supervision**.

Why weak supervision?  When it is easier to critique than to generate.

Basic template:

1. Generate responses from the model.

2. Score responses with a {human, verifier, LM judge}.

3. Update the model to prefer better responses.

Algorithms:

- Proximal Policy Optimization (PPO) from reinforcement learning  *[ppo_2017]* *[instruct_gpt_2022]*

- Direct Policy Optimization (DPO): for preference data, simpler  *[dpo_2023]*

- Group Relative Preference Optimization (GRPO): remove value function  *[grpo]*

Challenges:

- RL algorithms are unstable and hard to tune

- At scale, this requires a lot of new infrastructure (inference with async rollouts)

- Constantly trading off systems efficiency and on-policyness

## Assignment 5 (alignment)

 [GitHub](https://github.com/stanford-cs336/assignment5-alignment) [PDF from Spring 2025](https://github.com/stanford-cs336/assignment5-alignment/blob/spring2025/cs336_spring2025_assignment5_alignment.pdf)

- Implement Direct Preference Optimization (DPO)

- Implement Group Relative Preference Optimization (GRPO)

Remember it's all about **efficiency**:

- Resources: data + hardware (compute, memory, communication bandwidth)

- How do you train the best model given a fixed set of resources?

Today, we are compute-constrained, so design decisions will reflect squeezing the most out of given hardware.

- Systems: clearly about efficiency

- Tokenization: working with raw bytes is elegant, but compute-inefficient with today's model architectures

- Model architecture: many changes motivated by reducing memory or FLOPs (e.g., sharing KV caches, sliding window attention)

- Data filtering: avoid wasting precious compute updating on bad / irrelevant data

- Scaling laws: use less compute on smaller models to do hyperparameter tuning

Tomorrow, we will become data-constrained...

This unit was inspired by Andrej Karpathy's video on tokenization; check it out!  [https://www.youtube.com/watch?v=zduSFxRajkE](https://www.youtube.com/watch?v=zduSFxRajkE)

Raw text is generally represented as Unicode strings.

In [ ]:
string = "Hello, 🌍! 你好!"

A language model places a probability distribution over sequences of tokens (usually represented by integer indices).

In [ ]:
indices = [15496, 11, 995, 0]

So we need a procedure that *encodes* strings into tokens.

We also need a procedure that *decodes* tokens back into strings.

A  *[Tokenizer]* is a class that implements the encode and decode methods.

To get a feel for how tokenizers work, play with this  [interactive site](https://tiktokenizer.vercel.app/?encoder=gpt2)

## Observations

- A word and its preceding space are part of the same token (e.g., " world").

- A word at the beginning and in the middle are represented differently (e.g., "hello hello").

- Numbers are tokenized into every few digits.

Here's the GPT-5 tokenizer from OpenAI (tiktoken) in action.

In [ ]:
tokenizer = get_gpt5_tokenizer()  # @stepover

string = "Hello, 🌍! 你好!"  # @inspect string

Check that encode() and decode() roundtrip:

In [ ]:
indices = tokenizer.encode(string)  # @inspect indices

reconstructed_string = tokenizer.decode(indices)  # @inspect reconstructed_string

assert string == reconstructed_string

Compression ratio: number of bytes per token

In [ ]:
compression_ratio = get_compression_ratio(string, indices)  # @inspect compression_ratio

The larger the compression ratio, the shorter the sequence (good since attention is quadratic in sequence length).

One could increase compression ratio by increasing **vocabulary size** (number of possible token values increases), leading to sparsity.

In [ ]:
vocabulary_size = tokenizer.n_vocab  # @inspect vocabulary_size

Let's take a look at the actual vocabulary: 

In [ ]:
output_tokenizer(tokenizer, "var/gpt5_tokenizer_vocab.txt")  # @stepover

A Unicode string is a sequence of Unicode characters.

Each character can be converted into a code point (integer) via `ord`.

In [ ]:
assert ord("a") == 97

assert ord("🌍") == 127757

It can be converted back via `chr`.

In [ ]:
assert chr(97) == "a"

assert chr(127757) == "🌍"

Now let's build a `Tokenizer` and make sure it round-trips:

In [ ]:
tokenizer = CharacterTokenizer()

string = "Hello, 🌍! 你好!"  # @inspect string

indices = tokenizer.encode(string)  # call ord @inspect indices @stepover

reconstructed_string = tokenizer.decode(indices)  # call chr @inspect reconstructed_string @stepover

assert string == reconstructed_string

There are approximately 150K Unicode characters.  [Wikipedia](https://en.wikipedia.org/wiki/List_of_Unicode_characters)

In [ ]:
vocabulary_size = max(indices) + 1  # This is a lower bound @inspect vocabulary_size

Problem 1: this is a very large vocabulary.

Problem 2: many characters are quite rare (e.g., 🌍), which is inefficient use of the vocabulary.

In [ ]:
compression_ratio = get_compression_ratio(string, indices)  # @inspect compression_ratio @stepover

This tokenizer is the worst of both worlds (large vocabulary, low compression ratio).

Unicode strings can be represented as a sequence of bytes, which can be represented by integers between 0 and 255.

The most common Unicode encoding is  [UTF-8](https://en.wikipedia.org/wiki/UTF-8)

Some Unicode characters are represented by one byte:

In [ ]:
assert bytes("a", encoding="utf-8") == b"a"

Others take multiple bytes:

In [ ]:
assert bytes("🌍", encoding="utf-8") == b"\xf0\x9f\x8c\x8d"

Now let's build a `Tokenizer` and make sure it round-trips:

In [ ]:
tokenizer = ByteTokenizer()

string = "Hello, 🌍! 你好!"  # @inspect string

indices = tokenizer.encode(string)  # @inspect indices @stepover

reconstructed_string = tokenizer.decode(indices)  # @inspect reconstructed_string @stepover

assert string == reconstructed_string

The vocabulary is nice and small: a byte can represent 256 values.

In [ ]:
vocabulary_size = 256  # @inspect vocabulary_size

What about the compression rate?

In [ ]:
compression_ratio = get_compression_ratio(string, indices)  # @inspect compression_ratio @stepover

assert compression_ratio == 1

The compression ratio is terrible, which means the sequences will be too long.

Given that the context length of a Transformer is limited (since attention is quadratic), this is not looking great...

Another approach (closer to what was done classically in NLP) is to split strings into words.

In [ ]:
string = "I'll say supercalifragilisticexpialidocious!"

chunks = regex.findall(r"\w+|.", string)  # @inspect chunks

This regular expression keeps all alphanumeric characters together (words).

To turn this into a `Tokenizer`, we need to map these chunks into integers.

Then, we can build a mapping from each chunk into an integer.

What's good: each token is meaningful (since humans invented words).

In [ ]:
vocabulary_size = "Number of distinct chunks in the training data"

compression_ratio = get_compression_ratio(string, chunks)  # @inspect compression_ratio @stepover

Compression ratio is good, but vocabulary size can be huge.

Moreover:

- Many words are rare and the model won't learn much about them.

- This doesn't obviously provide a fixed vocabulary size.

- New words we haven't seen during training get a special UNK token, which is ugly and can mess up perplexity calculations.

## Byte Pair Encoding (BPE)

The BPE algorithm was introduced by Philip Gage in 1994 for data compression.  [http://www.pennelynn.com/Documents/CUJ/HTML/94HTML/19940045.HTM](http://www.pennelynn.com/Documents/CUJ/HTML/94HTML/19940045.HTM)

It was adapted to NLP for neural machine translation.  *[sennrich_2016]*

(Previously, papers had been using word-based tokenization.)

BPE was then used by GPT-2.  *[gpt2_2019]*

Basic idea: *train* the tokenizer on raw text to construct a vocabulary tailored to the data.

Intuition: common sequences of bytes are represented by a single token, rare sequences are represented by many tokens.

Sketch: start with each byte as a token, and successively merge the most common pair of adjacent tokens.

## Training the tokenizer

In [ ]:
string = "the cat in the hat"  # @inspect string

Start with the list of bytes of `string`.

In [ ]:
params = train_bpe(string, num_merges=3)

## Using the tokenizer

Now, given a new text, we can encode it.

In [ ]:
tokenizer = BPETokenizer(params)  # @stepover

string = "the quick brown fox"  # @inspect string

indices = tokenizer.encode(string)  # @inspect indices

reconstructed_string = tokenizer.decode(indices)  # @inspect reconstructed_string @stepover

assert string == reconstructed_string

In Assignment 1, you will go beyond this in the following ways:

- encode() currently loops over all merges. Only loop over merges that matter.

- Detect and preserve special tokens (e.g., <|endoftext|>).

- Use pre-tokenization (e.g., the GPT-2 tokenizer regex).

- Try to make the implementation as fast as possible.

Summary:

- Tokenizer: strings ↔ tokens (indices)

- Character-based, byte-based, word-based tokenization are highly suboptimal

- BPE is an effective heuristic that is data-driven

- Tokenization is a separate step, maybe one day do it end-to-end from bytes...

But whatever solution needs to satisfy:

1. Model (e.g., Transformer) should operate on chunks (abstractions) of the sequence (text, video, DNA, etc.)

2. Chunks should be variable (allocate more model capacity to interesting chunks)

Next time: resource accounting